Assigniment : NLP Project

Atomcamp

Daanish Khurshid

Step 1 : Define Your Use Case

Project Description: Central Bank Sentiment Analysis

I am developing a project to analyze central bank press releases in order to decode their underlying economic sentiment. Specifically, I want to determine whether a central bank views the economy as healthy—signaling a bullish or hawkish sentiment—or if they believe the economy is struggling, signaling a bearish or dovish sentiment. I will also identify when central bankers maintain a neutral stance.

Why This Matters

Assessing how central banks position themselves is crucial for traders. It allows them to align their trades with capital flows; as interest rates fluctuate, currency values and stock markets shift accordingly.

While central bank statements are released monthly and quarterly for individual traders to analyze, reviewing historical data over several years involves an overwhelming amount of reading. This is especially true given the number of major, actively traded currencies, which include the US Dollar (USD), British Pound (GBP), Canadian Dollar (CAD), New Zealand Dollar (NZD), Australian Dollar (AUD), Euro (EUR), Japanese Yen (JPY), and Swiss Franc (CHF).

Project Scope and Future Work

To automate this time-consuming process, this project will use text data to read historical press releases and decode their exact sentiment.

To keep the initial scope manageable, I will focus on only two or three central banks rather than all of them. The system will take a raw press statement as the input and generate the corresponding market sentiment (Hawkish/Bullish, Dovish/Bearish, or Neutral) as the output.

In future phases of this project, I plan to expand this sentiment analysis to social media platforms like X (formerly Twitter) and trading forums. This will allow me to compare retail trader sentiment on the street with the official stance of the central banks to see if they align or diverge.



In [9]:
# Installing Libraries

!pip install torch numpy pandas matplotlib scikit-learn

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 1.5 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 2.1 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 1.9 MB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 2.0 MB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 3.3 MB/s  0:00:02m0:00:0100:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 2.1 MB/s  0:00:01 eta 0:00:01
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 3.8 MB/s  0:00:01 eta 0:00:01
Using cached pyparsing-3.3.2-py3-none

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk

In [ ]:
# This is to check if i have extra computing power rather than the basic CPU
import torch

if torch.backends.mps.is_available():
    print("M2 Graphic Accelerator Detected! You can test your models locally.")
else:
    print("Running on basic CPU mode.")


/Users/daanishkhurshid/miniconda3/envs/nlp-env/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:295: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729646995093/work/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


M2 Graphic Accelerator Detected! You can test your models locally.


Downloading press statements from the FED (Fedenal Reserve Bank) USA, using Beautiful Soup 
These are FOMC statements

In [4]:
# Importing Library to run beautifulsoup4

!pip install beautifulsoup4 requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp312-cp312-macosx_10_13_universal2.whl.metadata (40 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.4.7-cp312-cp312-macosx_10_13_universal2.whl (311 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [requests]


In [5]:
import os                   # Helps python to access the operating system
import time                 # Allow us to add a delay between requests/downlaods to avoid IP blocking
import requests             # Helps python to make HTTP requests, acting like an automated browser
from bs4 import BeautifulSoup



In [7]:
import os

# Cfind the exact folder your notebook file is sitting in
notebook_dir = os.path.dirname(os.path.abspath('__file__'))

# This glues your desired folder path onto that exact location
output_folder = os.path.join(notebook_dir, "data", "fed_statements")

# This safely builds the folder structure on your Mac
os.makedirs(output_folder, exist_ok=True)

print(f"📁 Your files will save perfectly to: {output_folder}")


📁 Your files will save perfectly to: /Users/daanishkhurshid/Atomcamp /NLP Assignment/data/fed_statements


In [ ]:
# Creating folder where data will be saved
output_folder = "data/fed_statements"
os.makedirs(output_folder, exist_ok=True)


In [ ]:
base_url = "https://federalreserve.gov"
calendar_url = "https://federalreserve.gov/monetarypolicy/fomccalendars.htm"    # This is the URL of the page we want to scrape

response = requests.get(calendar_url)
soup = BeautifulSoup(response.text, "html.parser")


In [12]:
# code looks for every <a> tag on the page and stores the href attribute in the 'links' list
statement_links = []
# Updated loop to look for the exact Minutes URL structure you found
for link in soup.find_all("a", href=True):
    href = link["href"]
    
    if "/monetarypolicy/fomcminutes" in href and href.endswith(".htm"):
        full_url = base_url + href if href.startswith("/") else href
        
        if full_url not in statement_links:
            statement_links.append(full_url)


print("Number of Press Releases:", len(statement_links))    # This will print the number of links found on the page



Number of Press Releases: 43


In [13]:
# Processes each link

for url in statement_links[:5]: 
    res = requests.get(url)
    page_soup = BeautifulSoup(res.text, "html.parser")
    article_div = page_soup.find("div", id="article")


In [14]:
if article_div:
    # 1. Extract the clean text directly from the container
    text_content = article_div.get_text(separator="\n", strip=True)
    
    # 2. Extract the 8-digit date string automatically from the URL string
    date_str = "".join([c for c in url if c.isdigit()])[-8:] 
    
    # 3. Create the filepath and write the data inside a clean context block
    filepath = os.path.join(output_folder, f"fed_minutes_{date_str}.txt")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text_content)
        
    # 4. Take a polite parsing break
    time.sleep(1)



In [ ]:
# Scrapping Data from Federal Reserve
# Creating folder where data will be saved
output_folder = "data/fed_statements"
os.makedirs(output_folder, exist_ok=True)

base_url = "https://www.federalreserve.gov"
calendar_url = "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm"  # This is the URL of the page we want to scrape

response = requests.get(calendar_url)
soup = BeautifulSoup(response.text, "html.parser")

# code looks for every <a> tag on the page and stores the href attribute in the 'links' list
statement_links = []
# Updated loop to look for the exact Minutes URL structure you found
for link in soup.find_all("a", href=True):
    href = link["href"]
    
    if "/monetarypolicy/fomcminutes" in href and href.endswith(".htm"):
        full_url = base_url + href if href.startswith("/") else href
        
        if full_url not in statement_links:
            statement_links.append(full_url)

print("Number of Press Releases:", len(statement_links))  # This will print the number of links found on the page


for url in statement_links: 
    print(f"📥 Fetching: {url}")
    res = requests.get(url)
    page_soup = BeautifulSoup(res.text, "html.parser")
    article_div = page_soup.find("div", id="article")
    
    if article_div:
        # Extract the clean text directly from the container
        text_content = article_div.get_text(separator="\n", strip=True)
        
        # Extract the 8-digit date string automatically from the URL string
        date_str = "".join([c for c in url if c.isdigit()])[-8:] 
        
        # Create the filepath and write the data inside a clean context block
        filepath = os.path.join(output_folder, f"fed_minutes_{date_str}.txt")
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(text_content)
            
        print(f"   ✅ Saved: fed_minutes_{date_str}.txt")
        
        # 4. Take a polite parsing break
        time.sleep(1)


Number of Press Releases: 43
📥 Fetching: https://www.federalreserve.gov/monetarypolicy/fomcminutes20260128.htm
   ✅ Saved: fed_minutes_20260128.txt
📥 Fetching: https://www.federalreserve.gov/monetarypolicy/fomcminutes20260318.htm
   ✅ Saved: fed_minutes_20260318.txt
📥 Fetching: https://www.federalreserve.gov/monetarypolicy/fomcminutes20260429.htm
   ✅ Saved: fed_minutes_20260429.txt
📥 Fetching: https://www.federalreserve.gov/monetarypolicy/fomcminutes20250129.htm
   ✅ Saved: fed_minutes_20250129.txt
📥 Fetching: https://www.federalreserve.gov/monetarypolicy/fomcminutes20250319.htm
   ✅ Saved: fed_minutes_20250319.txt
📥 Fetching: https://www.federalreserve.gov/monetarypolicy/fomcminutes20250507.htm
   ✅ Saved: fed_minutes_20250507.txt
📥 Fetching: https://www.federalreserve.gov/monetarypolicy/fomcminutes20250618.htm
   ✅ Saved: fed_minutes_20250618.txt
📥 Fetching: https://www.federalreserve.gov/monetarypolicy/fomcminutes20250730.htm
   ✅ Saved: fed_minutes_20250730.txt
📥 Fetching: https:/

In [22]:
!pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 1.8 MB/s  0:00:04m0:00:0100:01


In [28]:


# Disable insecure request warnings caused by bypassing SSL certificate checks
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Establish path roots on your Mac
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
output_folder = os.path.join(notebook_dir, "data", "ecb_statements")
os.makedirs(output_folder, exist_ok=True)

# Standard browser agent headers to ensure smooth server handshakes
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)'}

# Target ECB monetary policy release links
ecb_urls = [
    "https://www.ecb.europa.eu/press/pr/date/2026/html/ecb.mp260430~81b7179e6f.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2026/html/ecb.mp260319~3057739775.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2026/html/ecb.mp260205~001d26959b.en.html", 
    "https://www.ecb.europa.eu/press/pr/date/2025/html/ecb.mp251218~58b0e415a6.en.html", 
    "https://www.ecb.europa.eu/press/pr/date/2025/html/ecb.mp251030~cf0540b5c0.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2025/html/ecb.mp250911~6afb7a9490.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2025/html/ecb.mp250724~50bc70e13f.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2025/html/ecb.mp250605~3b5f67d007.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2025/html/ecb.mp250417~42727d0735.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2025/html/ecb.mp250306~d4340800b3.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2025/html/ecb.mp250130~530b29e622.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.mp241212~2acab6e51e.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.mp241017~aa366eaf20.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.mp240912~67cb23badb.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.mp240718~b9e0ddd9d5.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.mp240606~2148ecdb3c.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.mp240411~1345644915.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.mp240411~1345644915.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2024/html/ecb.mp240125~f738889bde.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.mp231214~9846e62f62.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.mp231026~6028cea576.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.mp230914~aab39f8c21.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.mp230727~da80cfcf24.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.mp230615~d34cddb4c6.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.mp230504~cdfd11a697.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.mp230316~aad5249f30.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2023/html/ecb.mp230202~08a972ac76.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2022/html/ecb.mp221215~f3461d7b6e.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2022/html/ecb.mp221027~df1d778b84.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2022/html/ecb.mp220908~c1b6839378.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2022/html/ecb.mp220721~53e5bdd317.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2022/html/ecb.mp220609~122666c272.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2022/html/ecb.mp220414~d1b76520c6.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2022/html/ecb.mp220310~2d19f8ba60.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2022/html/ecb.mp220203~90fbe94662.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2021/html/ecb.mp211216~1b6d3a1fd8.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2021/html/ecb.mp211028~85474438a4.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2021/html/ecb.mp210909~2c94b35639.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2021/html/ecb.mp210722~48dc3b436b.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2021/html/ecb.mp210610~b4d5381df0.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2021/html/ecb.mp210422~f075ebe1f0.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2021/html/ecb.mp210311~35ba71f535.en.html",
    "https://www.ecb.europa.eu/press/pr/date/2021/html/ecb.mp210121~eb9154682e.en.html"
]

# Eliminate accidental duplicate urls inside the array while maintaining index order
ecb_urls = list(dict.fromkeys(ecb_urls))

print(f"Processing {len(ecb_urls)} clean ECB links...")

for url in ecb_urls:
    try:
        # Isolate the exact file tag component from the end of the URL text string
        url_slug = url.strip("/").split("/")[-1].replace(".en.html", "").replace(".html", "")
        
        # Parse the 6-digit date segment from the slug text and prefix it with '20' for standard YYYYMMDD format
        raw_date = url_slug.split("~")[0].replace("ecb.mp", "")
        formatted_date = "20" + raw_date
        
        print(f"Fetching statement for date: {formatted_date}")
        
        # Request data stream from the target node
        res = requests.get(url, headers=headers, verify=False)
        page_soup = BeautifulSoup(res.text, "html.parser")
        
        # Target main content layouts used across ECB frameworks
        content_container = page_soup.find("main") or page_soup.find("article")
        
        if content_container:
            # Extract paragraphs clean of structural code markers
            text_content = content_container.get_text(separator="\n", strip=True)
            
            # Map out file path using structured date keys
            filepath = os.path.join(output_folder, f"ecb_statement_{formatted_date}.txt")
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(text_content)
                
            print(f"   Saved: ecb_statement_{formatted_date}.txt")
            
            # Moderate request sequence pacing to prevent network timeout drops
            time.sleep(1)
        else:
            print(f"   Warning: Content element container missing for {url_slug}")
            
    except Exception as e:
        print(f"   Error connection dropped on link: {e}")

print("\nAll tasks finished. Check your data/ecb_statements directory.")


Processing 42 clean ECB links...
Fetching statement for date: 20260430
   Saved: ecb_statement_20260430.txt
Fetching statement for date: 20260319
   Saved: ecb_statement_20260319.txt
Fetching statement for date: 20260205
   Saved: ecb_statement_20260205.txt
Fetching statement for date: 20251218
   Saved: ecb_statement_20251218.txt
Fetching statement for date: 20251030
   Saved: ecb_statement_20251030.txt
Fetching statement for date: 20250911
   Saved: ecb_statement_20250911.txt
Fetching statement for date: 20250724
   Saved: ecb_statement_20250724.txt
Fetching statement for date: 20250605
   Saved: ecb_statement_20250605.txt
Fetching statement for date: 20250417
   Saved: ecb_statement_20250417.txt
Fetching statement for date: 20250306
   Saved: ecb_statement_20250306.txt
Fetching statement for date: 20250130
   Saved: ecb_statement_20250130.txt
Fetching statement for date: 20241212
   Saved: ecb_statement_20241212.txt
Fetching statement for date: 20241017
   Saved: ecb_statement_20241